# PadelVision — 04 Train the ball detector (Kaggle)

Fine-tunes **YOLO11s** (COCO-pretrained, the bake-off baseline) into a one-class ball detector, then scores it
against the same pooled labels as notebook 03.

**Training data**
1. `Test_video.mp4`: classical pseudo-labels (`ml/datasets/Test_video_ball_pseudo.csv`) + visually confirmed balls,
   **train blocks only**. The video is cut into 20 s blocks; odd blocks are test-only, and training frames keep
   a 2 s margin from them, so no near-duplicate frame is in both train and test.
2. Roboflow Universe padel-ball datasets, downloaded straight into this session (ball classes only).

**Test:** confirmed labels in the test blocks (~91 balls, 140 frames), with the COCO baseline re-run on the same frames.
Baselines there: COCO sports ball F1 0.64; Roboflow nazbq F1 0.68 (on its 68 frames).

**Setup (Kaggle):** GPU **T4 x2** (or P100), **Internet on**, video attached, secret `ROBOFLOW_API_KEY`.
Training takes roughly 1–3 h depending on the datasets enabled, so use **Save Version → Save & Run All (Commit)**.
It then runs in the background, and the outputs (weights + `ball_train_results.zip`) are kept with the version.

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} workdir={WORK}")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: enable one in the notebook settings"

In [ ]:
REPO_URL = "https://github.com/Duggineniakhil/PadelVision.git"
BRANCH = "v2-rebuild"
REPO = f"{WORK}/PadelVision"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!pip install -q -e "{REPO}/pipeline[ml]"

# Editable installs aren't picked up by an already-running kernel; make the package importable now.
sys.path.insert(0, f"{REPO}/pipeline/src")
import padelvision
print("padelvision", padelvision.__version__)

In [ ]:
!cd {REPO} && python scripts/fetch_models.py

In [ ]:
from pathlib import Path

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Leave empty to use the first video found under /kaggle/input (Kaggle) or MyDrive/padel (Colab).
VIDEO_PATH = ""

if not VIDEO_PATH:
    search_root = Path("/kaggle/input") if ON_KAGGLE else Path("/content/drive/MyDrive/padel")
    videos = sorted(p for ext in ("*.mp4", "*.mov", "*.avi", "*.mkv") for p in search_root.rglob(ext))
    print("Videos found:", *videos, sep="\n  ")
    assert videos, f"No video found under {search_root}; attach one or set VIDEO_PATH"
    VIDEO_PATH = str(videos[0])

assert os.path.exists(VIDEO_PATH), f"Not found: {VIDEO_PATH}"
print("Using video:", VIDEO_PATH)

In [ ]:
# ---- configuration -------------------------------------------------------------------
UNIVERSE = [  # (workspace, project, version, enabled)
    ("padel-analysis-eulgl", "padel-ball-detection-nazbq", 4, True),   # bake-off winner's data (~1k imgs)
    ("padel-ll7pp", "padel-dataset", 1, True),                         # ~9.2k imgs
    ("awecesar", "padel-ball-detection-i1vvv", 3, False),              # ~8.8k imgs (noisier model; off by default)
]
EPOCHS = 60
IMGSZ = 1280      # the ball is ~5-30 px: keep full resolution
BATCH = 8         # per GPU-set; fits a T4 (16 GB) at 1280 px with yolo11s
BLOCK_S = 20.0    # train/test block length (seconds)
SEED = 0
DS = Path(f"{WORK}/ball_ds")

## 1. Our video (train blocks)

In [ ]:
import json, shutil
import cv2
import numpy as np
import pandas as pd
from padelvision.ball.dataset import (count, import_yolo_dataset, is_test_frame,
                                      is_train_frame, write_data_yaml, write_video_frames)
from padelvision.ball.evaluate import curve, review_crops

fps = cv2.VideoCapture(VIDEO_PATH).get(cv2.CAP_PROP_FPS)
LABELS = pd.read_csv(f"{REPO}/ml/eval/labels/Test_video_ball.csv")
EVAL = json.loads(Path(f"{REPO}/ml/eval/labels/Test_video_ball_frames.json").read_text())
EVAL_FRAMES = [f for f in EVAL["frames"] if f not in set(EVAL["excluded"])]
TEST_FRAMES = [f for f in EVAL_FRAMES if is_test_frame(f, fps, BLOCK_S)]

pseudo = pd.read_csv(f"{REPO}/ml/datasets/Test_video_ball_pseudo.csv")
pseudo = pseudo[is_train_frame(pseudo.frame, fps, BLOCK_S)]
confirmed = LABELS[(LABELS.status == "ball") & is_train_frame(LABELS.frame, fps, BLOCK_S)]
confirmed = confirmed.assign(size=16.0)
# Frames with a visual review use the reviewed balls (all of them, incl. spare balls) instead of pseudo-labels.
reviewed_frames = set(LABELS.frame)
boxes = pd.concat([pseudo[~pseudo.frame.isin(reviewed_frames)][["frame", "u", "v", "size"]],
                   confirmed[["frame", "u", "v", "size"]]])
frames = sorted(boxes.frame.unique())
rng = np.random.default_rng(SEED)
val = set(rng.choice(frames, max(1, len(frames) // 10), replace=False))

if DS.exists():
    shutil.rmtree(DS)
n_tr = write_video_frames(VIDEO_PATH, boxes[~boxes.frame.isin(val)], DS, "train", "vid")
n_va = write_video_frames(VIDEO_PATH, boxes[boxes.frame.isin(val)], DS, "val", "vid")
SOURCES = [{"source": "Test_video.mp4 (train blocks)", "images": n_tr + n_va, "balls": len(boxes)}]
print(f"own video: {n_tr} train + {n_va} val frames, {len(boxes)} balls; test frames held out: {len(TEST_FRAMES)}")

## 2. Roboflow Universe datasets

In [ ]:
!pip install -q roboflow
from roboflow import Roboflow
try:
    from kaggle_secrets import UserSecretsClient
    RF_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
except Exception:
    RF_KEY = os.environ.get("ROBOFLOW_API_KEY")

if not RF_KEY:
    print("No ROBOFLOW_API_KEY secret: training on our video only")
else:
    rf = Roboflow(api_key=RF_KEY)
    for ws, proj, ver, enabled in UNIVERSE:
        if not enabled:
            continue
        try:
            version = rf.workspace(ws).project(proj).version(ver)
            dl = version.download("yolov8", location=f"{WORK}/universe/{proj}", overwrite=True)
            stats = import_yolo_dataset(dl.location, DS, prefix=proj)
            readme = next(Path(dl.location).glob("README*"), None)
            lic = [l.strip() for l in readme.read_text().splitlines() if "icense" in l] if readme else []
            stats["license"] = lic[0] if lic else "see Roboflow Universe page"
            stats["id"] = f"{ws}/{proj}/{ver}"
            SOURCES.append(stats)
            print(f"{proj}: {stats['images']} images, {stats['balls']} balls, "
                  f"kept classes {stats['kept_classes']} of {stats['classes']}; {stats['license']}")
        except Exception as e:
            print(f"{proj} failed:", repr(e)[:300])

DATA = write_data_yaml(DS, SOURCES)
print("dataset:", count(DS))

## 3. Train

In [ ]:
import torch
from ultralytics import YOLO
from padelvision.models import weights_path

model = YOLO(str(weights_path("person-detector")))  # yolo11s COCO weights
devices = [0, 1] if torch.cuda.device_count() > 1 else 0
model.train(
    data=str(DATA), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=devices,
    project=f"{WORK}/train", name="ball", exist_ok=True, seed=SEED,
    patience=15, cos_lr=True, close_mosaic=10, single_cls=True,
    degrees=0.0, fliplr=0.5, scale=0.3,  # mild scaling: don't shrink tiny balls away
    workers=4, plots=True,
)
BEST = Path(f"{WORK}/train/ball/weights/best.pt")
print("best weights:", BEST, BEST.exists())

## 4. Evaluate on the held-out test blocks

In [ ]:
from padelvision.models.ball_detectors import YoloBall

need = set(TEST_FRAMES)
FRAMES, cap, f = {}, cv2.VideoCapture(VIDEO_PATH), 0
while f <= max(need):
    ok, img = cap.read()
    if not ok:
        break
    if f in need:
        FRAMES[f] = img
    f += 1
cap.release()

OUT = Path(f"{WORK}/ball_train_results")
(OUT / "crops").mkdir(parents=True, exist_ok=True)
rows = {}
for name, det in [("trained_yolo11s", YoloBall(BEST, classes=[0], conf=0.05)),
                  ("coco_sports_ball", YoloBall(weights_path("person-detector"), classes=[32], conf=0.05))]:
    preds = det.detect(FRAMES, TEST_FRAMES)
    preds.to_parquet(OUT / f"preds_{name}.parquet")
    c = curve(preds, LABELS, TEST_FRAMES)
    c.to_csv(OUT / f"curve_{name}.csv", index=False)
    rows[name] = c.loc[c.f1.astype(float).idxmax()].to_dict()
    review_crops(preds, LABELS, FRAMES, OUT / "crops", name, max_crops=60)
table = pd.DataFrame(rows).T
display(table)
table.to_csv(OUT / "results.csv")

**Reading the table:** each row uses its best confidence threshold. `unverified` = detections on spots nobody has
reviewed yet (crops in the zip). Those count as neither right nor wrong until reviewed, so a model with many
unverified detections may move after review.

## 5. Save weights + results

In [ ]:
import hashlib
shutil.copy(BEST, OUT / "padel_ball_yolo11s.pt")
sha = hashlib.sha256((OUT / "padel_ball_yolo11s.pt").read_bytes()).hexdigest()
for extra in ["results.csv", "results.png", "confusion_matrix.png", "args.yaml"]:
    p = Path(f"{WORK}/train/ball/{extra}")
    if p.exists():
        shutil.copy(p, OUT / f"train_{extra}")
(OUT / "model_card.json").write_text(json.dumps({
    "file": "padel_ball_yolo11s.pt", "sha256": sha, "base": "yolo11s.pt (COCO)",
    "imgsz": IMGSZ, "epochs": EPOCHS, "block_s": BLOCK_S, "sources": SOURCES,
    "test": {"frames": len(TEST_FRAMES), "results": rows},
}, indent=1, default=str))
print("sha256:", sha)

from IPython.display import FileLink
ZIP = f"{WORK}/ball_train_results.zip"
shutil.make_archive(ZIP[:-4], "zip", OUT.parent, OUT.name)
print(f"{ZIP}: {Path(ZIP).stat().st_size / 1e6:.1f} MB")
FileLink(Path(ZIP).relative_to(Path.cwd()) if Path(ZIP).is_relative_to(Path.cwd()) else ZIP)

### Optional: publish the weights to the Hugging Face Hub
Add a Kaggle secret `HF_TOKEN` (a write token) and set `HF_REPO` (e.g. `your-name/padelvision-models`).
The weights then get a permanent download URL for `models/manifest.yaml`. Skip this cell otherwise.

In [ ]:
HF_REPO = ""  # e.g. "your-hf-username/padelvision-models"
if HF_REPO:
    !pip install -q huggingface_hub
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HF_REPO, repo_type="model", exist_ok=True, private=False)
    api.upload_file(path_or_fileobj=str(OUT / "padel_ball_yolo11s.pt"),
                    path_in_repo="padel_ball_yolo11s.pt", repo_id=HF_REPO)
    api.upload_file(path_or_fileobj=str(OUT / "model_card.json"),
                    path_in_repo="padel_ball_yolo11s.json", repo_id=HF_REPO)
    print(f"https://huggingface.co/{HF_REPO}/resolve/main/padel_ball_yolo11s.pt")